In [ ]:
import os
import re
import glob
import csv
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import pdfplumber
import fitz  # PyMuPDF

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ----------------------------
# Config
# ----------------------------
DEFAULT_SIM_THRESHOLD = 0.78
MAX_PAGES_TO_SAMPLE = 1
MAX_LINES = 80
DROP_NUMBER_HEAVY_LINES = True


@dataclass
class PdfTemplateFingerprint:
    path: str
    fingerprint_text: str


# ============================
# Text normalization
# ============================
def _normalize_line(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)

    # Remove variable elements
    s = re.sub(r"\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b", " <DATE> ", s)
    s = re.sub(r"\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b", " <DATE> ", s)
    s = re.sub(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", " <EMAIL> ", s, flags=re.I)

    s = re.sub(r"₹", " INR ", s)
    s = re.sub(r"\bINR\b", " INR ", s, flags=re.I)
    s = re.sub(r"\b\d{6,}\b", " <LONGNUM> ", s)
    s = re.sub(r"\b\d+\.\d+\b", " <DECIMAL> ", s)
    s = re.sub(r"\b\d+\b", " <NUM> ", s)

    s = re.sub(r"[^a-zA-Z<>/ _:-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s


def _is_number_heavy(line: str) -> bool:
    raw = re.sub(r"\s+", "", line)
    if not raw:
        return True
    digits = sum(ch.isdigit() for ch in raw)
    return digits / max(1, len(raw)) > 0.45


# ============================
# 🔐 SECURITY CHECK (FITZ)
# ============================
def inspect_pdf_security(filepath: str) -> Tuple[bool, bool]:
    """
    Uses PyMuPDF (fitz) to detect:
      - is_password_protected
      - has_copy_permission
    """
    try:
        doc = fitz.open(filepath)

        # True if password required to open
        is_password_protected = bool(doc.needs_pass)

        # Copy permission check
        try:
            can_copy = bool(doc.permissions & fitz.PDF_PERM_COPY)
        except Exception:
            can_copy = False

        doc.close()
        return is_password_protected, can_copy

    except Exception:
        # If file cannot be opened
        return True, False


# ============================
# Template fingerprint
# ============================
def fingerprint_pdf(pdf_path: str,
                    max_pages: int = MAX_PAGES_TO_SAMPLE,
                    max_lines: int = MAX_LINES) -> PdfTemplateFingerprint:

    lines_out: List[str] = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages = pdf.pages[:max_pages]

            for p in pages:
                txt = p.extract_text() or ""
                raw_lines = txt.splitlines()[:max_lines]

                for ln in raw_lines:
                    if DROP_NUMBER_HEAVY_LINES and _is_number_heavy(ln):
                        continue

                    norm = _normalize_line(ln)
                    if norm and len(norm) >= 3:
                        lines_out.append(norm)

        # Deduplicate while preserving order
        seen = set()
        uniq = []
        for l in lines_out:
            if l not in seen:
                uniq.append(l)
                seen.add(l)

        fp_text = "\n".join(uniq)

    except Exception as e:
        fp_text = f"__ERROR__ {e}"

    return PdfTemplateFingerprint(path=pdf_path, fingerprint_text=fp_text)


# ============================
# Clustering
# ============================
def cluster_by_similarity(
    fps: List[PdfTemplateFingerprint],
    sim_threshold: float = DEFAULT_SIM_THRESHOLD,
) -> Tuple[List[int], np.ndarray]:

    docs = [fp.fingerprint_text for fp in fps]

    vec = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
    X = vec.fit_transform(docs)

    sim = cosine_similarity(X)
    n = sim.shape[0]

    adj: Dict[int, List[int]] = {i: [] for i in range(n)}

    for i in range(n):
        for j in range(i + 1, n):
            if sim[i, j] >= sim_threshold:
                adj[i].append(j)
                adj[j].append(i)

    labels = [-1] * n
    cluster_id = 0

    for i in range(n):
        if labels[i] != -1:
            continue

        stack = [i]
        labels[i] = cluster_id

        while stack:
            u = stack.pop()
            for v in adj[u]:
                if labels[v] == -1:
                    labels[v] = cluster_id
                    stack.append(v)

        cluster_id += 1

    return labels, sim


# ============================
# Main runner
# ============================
def classify_folder_to_csv(
    pdf_folder: str,
    output_csv_path: str,
    pattern: str = "*.pdf",
    sim_threshold: float = DEFAULT_SIM_THRESHOLD,
) -> str:

    pdf_paths = sorted(glob.glob(os.path.join(pdf_folder, pattern)))

    if not pdf_paths:
        raise FileNotFoundError(f"No PDFs found in: {pdf_folder}")

    fps = [fingerprint_pdf(p) for p in pdf_paths]
    labels, _ = cluster_by_similarity(fps, sim_threshold=sim_threshold)

    rows = []

    for fp, cluster in zip(fps, labels):
        folder = os.path.dirname(fp.path)
        filename = os.path.basename(fp.path)

        is_pw, can_copy = inspect_pdf_security(fp.path)

        rows.append([
            folder,
            filename,
            cluster,
            is_pw,
            can_copy,
        ])

    os.makedirs(os.path.dirname(output_csv_path) or ".", exist_ok=True)

    with open(output_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "folder",
            "filename",
            "cluster",
            "is_password_protected",
            "has_copy_permission",
        ])
        writer.writerows(rows)

    print(f"✅ Wrote: {output_csv_path}")
    print(f"Total PDFs: {len(pdf_paths)} | clusters: {len(set(labels))}")

    return output_csv_path


# ============================
# CLI entry
# ============================
if __name__ == "__main__":
    folder = os.environ.get("CAS_PDF_FOLDER", "")
    out_csv = os.environ.get("CAS_OUT_CSV", "cas_template_clusters.csv")

    if not folder:
        print("Set CAS_PDF_FOLDER env var.")
    else:
        classify_folder_to_csv(folder, out_csv)